# Phase 5 substrate snapshot capture (Colab)

**Purpose.** Capture Phase 4 substrate snapshots (memory + consolidation per scale, with positions) at three step points so Phase 5's schema-source robustness ablation (`phase-5-checklist.md` §C) can run against real post-death, pre-death, and step-1500 substrates.

**What gets saved.** Per seed × per scale × per step (default scale W=2, steps 500/1500/3000):
- `<output-dir>/snapshots/phase3_phase4_w2_step500.pt`
- `<output-dir>/snapshots/phase3_phase4_w2_step1500.pt`
- `<output-dir>/snapshots/phase3_phase4_w2_step3000.pt`

Each `.pt` file holds `(patterns, consolidation state, positions, config, metadata)` and round-trips through `energy_memory.phase4.snapshot.load_substrate_snapshot`. Phase 5's `experiments/40_phase5_branching.py --substrate-snapshot <path>` reads these directly.

**Configuration.** Defaults match Phase 4's graduated regime: online Hebbian @ st=0.3, n_cues=3000, β=10, m=6, A_k off, freq-α off (the report 040 result; mass death already does the filtering work). Mirrors the freq-α sweep notebook structure.

**Wall time.** 5 seeds in parallel on Colab A100 ≈ 14 min.

In [ ]:
# 1. Clone the repo and verify Phase 5 snapshot code is present.
%cd /content
!rm -rf Neuro-AI
!git clone https://github.com/Dypatterson/Neuro-AI.git
%cd Neuro-AI
!git log --oneline -5

import subprocess
for marker, fpath, label in [
    ('def save_substrate_snapshot', 'src/energy_memory/phase4/snapshot.py', 'snapshot save'),
    ('--snapshot-steps',             'experiments/19_phase34_integrated.py', 'exp 19 CLI'),
    ('positions=slot_positions',     'experiments/19_phase34_integrated.py', 'positions in snapshot'),
]:
    r = subprocess.run(['grep', '-n', marker, fpath], capture_output=True, text=True)
    status = '✓' if r.returncode == 0 else '✗ MISSING'
    print(f'{status} {label}: {marker}')
    if r.stdout:
        print(f'  {r.stdout.strip().splitlines()[0]}')

In [ ]:
# 2. Mount Drive and stage the phase3c codebook.
from google.colab import drive
drive.mount('/content/drive')

import shutil, os
src = '/content/drive/MyDrive/neuro-ai/phase3c_codebook_reconstruction.pt'
dst_dir = 'reports/phase3c_reconstruction'
os.makedirs(dst_dir, exist_ok=True)
shutil.copy(src, f'{dst_dir}/phase3c_codebook_reconstruction.pt')
!ls -lh {dst_dir}/phase3c_codebook_reconstruction.pt

In [ ]:
# 3. Install deps.
!pip install -q datasets

In [ ]:
# 4. CPU-only sanity check. Parent must NOT touch CUDA so workers can grab the GPU.
import sys, os, gc
sys.path.insert(0, 'src')

# Verify the codebook loads on CPU.
from energy_memory.phase2.persistence import load_codebook
cb = load_codebook('reports/phase3c_reconstruction/phase3c_codebook_reconstruction.pt', device='cpu')
print('codebook (parent CPU load):', cb.shape, cb.dtype, cb.device)
del cb; gc.collect()

# Sanity-check the snapshot round-trip (CPU only).
from energy_memory.phase4.snapshot import save_substrate_snapshot, load_substrate_snapshot
from energy_memory.memory.torch_hopfield import TorchHopfieldMemory
from energy_memory.phase4.consolidation import ConsolidationConfig, ConsolidationState
from energy_memory.phase2.encoding import build_position_vectors
from energy_memory.substrate.torch_fhrr import TorchFHRR
import torch
torch.manual_seed(0)
sub = TorchFHRR(dim=64, device='cpu')
mem = TorchHopfieldMemory(sub)
cons = ConsolidationState(ConsolidationConfig(m=4), device='cpu')
for _ in range(3):
    p = sub.normalize(torch.randn(64, dtype=torch.complex64))
    mem.store(p); cons.add_pattern(novelty_strength=1.0)
pos = build_position_vectors(sub, count=2)
save_substrate_snapshot(memory=mem, consolidation=cons, path='/tmp/sanity_snap.pt', positions=pos)
_, _, info = load_substrate_snapshot(path='/tmp/sanity_snap.pt', substrate=sub)
assert info['positions'] is not None and info['positions'].shape == (2, 64), 'snapshot positions broken'
print('snapshot round-trip sanity check: PASS')

In [ ]:
# 4b. Pre-warm the wikitext cache so subprocesses don't race on download.
print('warming wikitext cache...')
from energy_memory.phase2.corpus import load_corpus_splits
from pathlib import Path
splits = load_corpus_splits('wikitext', Path('.'), wikitext_name='wikitext-2-raw-v1')
print('  train:', len(splits['train']), 'rows')
print('  validation:', len(splits['validation']), 'rows')
del splits; gc.collect()
print('cache warmed.')

In [ ]:
# 4c. GPU info (still no CUDA init in parent).
print('=== GPU info ===')
!nvidia-smi --query-gpu=name,memory.total,compute_mode --format=csv
print()
print('=== Current GPU processes (should be empty before launching workers) ===')
!nvidia-smi --query-compute-apps=pid,process_name,used_memory --format=csv

In [ ]:
# 5. Launch parallel workers. One per seed; each captures three snapshots
#    (steps 500, 1500, 3000) at scale W=2 in its own output directory.

SEEDS = [17, 11, 23, 1, 2]
SNAPSHOT_STEPS = '500,1500,3000'  # §C: pre-death, step-1500, post-death
SNAPSHOT_SCALES = '2'             # W=2 is the post-death small population
N_CUES = 3000                     # Phase 4 graduation regime
RUN_TAG = 'phase5_snapshots'

import subprocess, os, time, signal
from pathlib import Path

os.environ['PYTHONPATH'] = '/content/Neuro-AI/src'
log_root = Path(f'reports/{RUN_TAG}_colab')
log_root.mkdir(parents=True, exist_ok=True)

def launch(seed):
    out_dir = f'reports/{RUN_TAG}_seed{seed}'
    Path(out_dir).mkdir(parents=True, exist_ok=True)
    log_path = log_root / f'seed{seed}.log'
    logf = open(log_path, 'w')
    cmd = [
        'python', 'experiments/19_phase34_integrated.py',
        '--device', 'cuda',
        '--updater-kind', 'hebbian',
        '--seed', str(seed),
        '--n-cues', str(N_CUES),
        '--store-threshold', '0.3',  # st=0.3 per Phase 4 graduation
        '--snapshot-steps', SNAPSHOT_STEPS,
        '--snapshot-scales', SNAPSHOT_SCALES,
        '--output-dir', out_dir,
    ]
    print('launching seed', seed, '->', out_dir, 'log:', log_path)
    proc = subprocess.Popen(cmd, stdout=logf, stderr=subprocess.STDOUT, env=os.environ.copy())
    return proc, logf, log_path

procs = [launch(s) for s in SEEDS]

# Wait, polling.
t0 = time.time()
remaining = list(range(len(SEEDS)))
while remaining:
    still_alive = []
    for i in remaining:
        proc, logf, log_path = procs[i]
        rc = proc.poll()
        if rc is None:
            still_alive.append(i)
        else:
            logf.close()
            mins = (time.time() - t0) / 60
            print(f'seed {SEEDS[i]} done (rc={rc}) at {mins:.1f} min')
    remaining = still_alive
    if remaining:
        time.sleep(30)
print(f'ALL DONE in {(time.time()-t0)/60:.1f} min')

In [ ]:
# 5b. EMERGENCY kill if cell 5 misbehaves. Interrupt cell 5 first, then run this.
import subprocess, signal, os
killed = 0
for line in subprocess.check_output(['ps', '-eo', 'pid,cmd']).decode().splitlines():
    if 'experiments/19' in line and 'grep' not in line:
        try:
            pid = int(line.split()[0])
            os.kill(pid, signal.SIGKILL)
            print(f'  killed {pid}')
            killed += 1
        except Exception as e:
            print(f'  pid err: {e}')
print(f'killed {killed} workers')

In [ ]:
# 6. Per-seed snapshot inventory + summary.
import json
from pathlib import Path

SEEDS = [17, 11, 23, 1, 2]
RUN_TAG = 'phase5_snapshots'

for seed in SEEDS:
    snap_dir = Path(f'reports/{RUN_TAG}_seed{seed}/snapshots')
    if not snap_dir.exists():
        print(f'seed {seed}: NO SNAPSHOTS')
        continue
    snaps = sorted(snap_dir.glob('*.pt'))
    sizes = [p.stat().st_size / 1024 for p in snaps]
    print(f'seed {seed}: {len(snaps)} snapshots')
    for p, sz in zip(snaps, sizes):
        print(f'  {p.name:60s}  {sz:8.1f} KiB')
    # Peek at one to confirm shape.
    if snaps:
        import torch
        s = torch.load(snaps[-1], map_location='cpu', weights_only=False)
        n_pat = s['patterns'].shape[0]
        d = s['patterns'].shape[-1] if n_pat else None
        has_pos = s.get('positions') is not None
        meta = s.get('metadata', {})
        print(f'    n_patterns={n_pat}, dim={d}, positions={"yes" if has_pos else "no"}, meta={meta}')

In [ ]:
# 7. Copy snapshots back to Drive.
import shutil, os
SEEDS = [17, 11, 23, 1, 2]
RUN_TAG = 'phase5_snapshots'
dst = '/content/drive/MyDrive/neuro-ai/results'
os.makedirs(dst, exist_ok=True)
for seed in SEEDS:
    src = f'reports/{RUN_TAG}_seed{seed}'
    if os.path.isdir(src):
        shutil.copytree(src, f'{dst}/{RUN_TAG}_seed{seed}', dirs_exist_ok=True)
shutil.copytree(f'reports/{RUN_TAG}_colab', f'{dst}/{RUN_TAG}_colab', dirs_exist_ok=True)
print('snapshots copied to', dst)
!ls {dst} | grep -E 'phase5_snapshots' | head -20

In [ ]:
# 8. (Optional) Quick Phase 5 headline-mode smoke against ONE snapshot,
#    to confirm the loader + driver works on real Phase 4 data before
#    closing the Colab session.
SEED = 17
SNAP_PATH = f'reports/phase5_snapshots_seed{SEED}/snapshots/phase3_phase4_w2_step3000.pt'
!python experiments/40_phase5_branching.py \
    --mode headline --device cuda --seed {SEED} \
    --substrate-snapshot {SNAP_PATH} \
    --n-cues 50 --k 10 \
    --output-dir reports/phase5_headline_seed{SEED}_step3000

import json
result = json.load(open(f'reports/phase5_headline_seed{SEED}_step3000/phase5_headline_seed{SEED}.json'))
print('\n=== headline ΔE (paired, n_cues=50) ===')
for tag, info in (result.get('headline_deltas') or {}).items():
    print(f'  {tag}: mean ΔE_content - ΔE_role = {info["mean_delta_e_content_minus_role"]:+.4f}, '
          f'frac positive = {info["fraction_positive"]:.2f} (n={info["n_pairs"]})')
shutil.copytree(f'reports/phase5_headline_seed{SEED}_step3000',
                f'{dst}/phase5_headline_seed{SEED}_step3000', dirs_exist_ok=True)
print('headline smoke saved to Drive')